## Exploratory Analysis PT 2: Feature Engineering / Pre_Modeling 

#### About Dataset

***Source***: Kaggle - [Heart Disease Prediction Dataset]
(https://www.kaggle.com/datasets/nalisha/heart-disease-prediction-dataset/data)
#### Data Description :
- Includes 270 Patient Medical Records with 14 Features
- Features include Age, Sex, Chest Pain, BP, Cholesterol, FBS over 120, EKG Results, Max HR, Exercise Angina, ST Depression, 
Slope of ST, Number of Vessels Fluro, Thalium and the key Target variable indicating presence of Heart Disease.

- Goal is to predict presence of Heart Disease based on these features using classification models.

License CCO (public domain)

- This notebook performs an initial exploratory analysis of the heart disease dataset, focusing on understanding data quality, 
- structure, and key patterns. The goal is to inspect distributions, identify missing or anomalous values, and generate early 
- insights that will guide subsequent feature engineering and modeling steps

## Data Features Explained
### Sex
- 1 = male  
- 0 = female  
### Chest Pain Type
- 4 = Asymptomatic  
- 3 = Non-anginal chest pain  
- 2 = Atypical angina  
- 1 = Typical angina  
### FBS over 120 (Fasting Blood Sugar > 120)
- 1 = Yes  
- 0 = No  
### EKG Results
- 2 = Probable left ventricular hypertrophy  
- 1 = ST-T wave abnormality  
- 0 = Normal  
### Exercise Angina
- 1 = Yes  
- 0 = No  
### Slope of ST
- 3 = Downsloping: strong indicator of coronary heart disease  
- 2 = Flat/horizontal: moderate indicator of coronary heart disease (inadequate flow to the heart muscle)  
- 1 = Upsloping: lowest risk of coronary heart disease (normal response to stress)  
#### Number of Vessels Fluoro
- 0, 1, 2, or 3 vessels have narrowed  
### Thallium
- 3 = Normal blood flow  
- 6 = Fixed defect: reduced blood flow at rest and stress, usually indicating previous heart attack  
- 7 = Reversible defect: normal blood flow at rest and reduced flow during stress (ischemia, significant narrowing, high risk)  


In [76]:
## Import proper packages for session 

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from pathlib import Path

In [77]:
## Load the dataset 

df_heart_cleaned = pd.read_csv('../../data/interim/df_heart_cleaned_v1.csv')
df_heart_cleaned.head()

,age,sex,chest_pain_type,bp,cholesterol,fbs_over_120,ekg_results,max_hr,exercise_angina,st_depression,slope_of_st,number_of_vessels_fluro,thallium,heart_disease
0,70,1,4,130,322,0,2,109,0,2.4,2,3,3,Presence
1,67,0,3,115,564,0,2,160,0,1.6,2,0,7,Absence
2,57,1,2,124,261,0,0,141,0,0.3,1,0,7,Presence
3,64,1,4,128,263,0,0,105,1,0.2,2,1,7,Absence
4,74,0,2,120,269,0,2,121,1,0.2,1,1,3,Absence


In [ ]:
## map heart_disease as a binary option a

df_heart_cleaned['heart_disease_flag'] = (
    df_heart_cleaned['heart_disease'].map({'Absence': 0, 'Presence': 1})
)
df_heart_v2ohe = df_heart_cleaned.drop(columns=['heart_disease'])



In [ ]:
## save the the new dataframe into project folder

df_heart_v2ohe = pd.concat(
    [df_heart_cleaned.drop(columns=['heart_disease'])],
    axis=1
)

df_heart_v2ohe.to_csv('../../data/processed/df_heart_v2ohe.csv', index=False)


In [ ]:
## display the firat 10 rows 

df_heart_v2ohe.head(10)

,age,sex,chest_pain_type,bp,cholesterol,fbs_over_120,ekg_results,max_hr,exercise_angina,st_depression,slope_of_st,number_of_vessels_fluro,thallium,heart_disease_flag
0,70,1,4,130,322,0,2,109,0,2.4,2,3,3,1
1,67,0,3,115,564,0,2,160,0,1.6,2,0,7,0
2,57,1,2,124,261,0,0,141,0,0.3,1,0,7,1
3,64,1,4,128,263,0,0,105,1,0.2,2,1,7,0
4,74,0,2,120,269,0,2,121,1,0.2,1,1,3,0
5,65,1,4,120,177,0,0,140,0,0.4,1,0,7,0
6,56,1,3,130,256,1,2,142,1,0.6,2,1,6,1
7,59,1,4,110,239,0,2,142,1,1.2,2,1,7,1
8,60,1,4,140,293,0,2,170,0,1.2,2,2,7,1
9,63,0,4,150,407,0,2,154,0,4.0,2,3,7,1


In [ ]:
# set random_state , defined columns for pre-modeling 

random_state = 21

cat_cols = ['sex', 'chest_pain_type', 'fbs_over_120', 'ekg_results',
            'exercise_angina', 'slope_of_st', 'number_of_vessels_fluro', 'thallium']

num_cols = ['age', 'bp', 'cholesterol', 'max_hr', 'st_depression']

target_col = 'heart_disease_flag'


In [ ]:
## get fresh df info 

df_heart_v2ohe.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 270 entries, 0 to 269
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   age                      270 non-null    int64  
 1   sex                      270 non-null    int64  
 2   chest_pain_type          270 non-null    int64  
 3   bp                       270 non-null    int64  
 4   cholesterol              270 non-null    int64  
 5   fbs_over_120             270 non-null    int64  
 6   ekg_results              270 non-null    int64  
 7   max_hr                   270 non-null    int64  
 8   exercise_angina          270 non-null    int64  
 9   st_depression            270 non-null    float64
 10  slope_of_st              270 non-null    int64  
 11  number_of_vessels_fluro  270 non-null    int64  
 12  thallium                 270 non-null    int64  
 13  heart_disease_flag       270 non-null    int64  
dtypes: float64(1), int64(13)
m

In [ ]:
## check class balances and distribution to prep model choice
 
df_heart_v2ohe['heart_disease_flag'].value_counts(normalize=True) * 100



heart_disease_flag
0    55.555556
1    44.444444
Name: proportion, dtype: float64

In [ ]:
## check unique values for each categorical column 

df_heart_v2ohe[cat_cols].nunique()


sex                        2
chest_pain_type            4
fbs_over_120               2
ekg_results                3
exercise_angina            2
slope_of_st                3
number_of_vessels_fluro    4
thallium                   3
dtype: int64

In [80]:
## get statistics for numerical columns 

df_heart_v2ohe[num_cols].describe()

,age,bp,cholesterol,max_hr,st_depression
count,270.000000,270.000000,270.000000,270.000000,270.00000
mean,54.433333,131.344444,249.659259,149.677778,1.05000
std,9.109067,17.861608,51.686237,23.165717,1.14521
min,29.000000,94.000000,126.000000,71.000000,0.00000
25%,48.000000,120.000000,213.000000,133.000000,0.00000
50%,55.000000,130.000000,245.000000,153.500000,0.80000
75%,61.000000,140.000000,280.000000,166.000000,1.60000
max,77.000000,200.000000,564.000000,202.000000,6.20000


In [ ]:
## Train/Validate/Test Split with stratification (keep the proportion of the target column similar across splits due to small data size)

X = df_heart_v2ohe[cat_cols + num_cols]
y = df_heart_v2ohe[target_col]

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size = 0.3, stratify=y, random_state = random_state)

X_valid, X_test, y_valid, y_test = train_test_split(X_temp, y_temp, test_size = 0.5, stratify = y_temp, random_state = random_state)

In [82]:
## build a pre-processing pipeline 
## scale numeric features
## encode categorical features for modeling 
## Numeric pipeline: fill missing numeric values with median, then standardize scale. 
## Categorical pipeline: fill missing categories with the most frequent, then one‑hot encode, safely handling unseen levels.


numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)


categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ]
)

In [84]:
## build the full preproccesor 
## This object knows: “apply numeric pipeline to num_cols, categorical pipeline to cat_cols, then concatenate the results”.

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
    ]
)


In [85]:
## fit and inspect the transformed data 

preprocessor.fit(X_train)

X_train_pre = preprocessor.transform(X_train)
X_valid_pre = preprocessor.transform(X_valid)
X_test_pre  = preprocessor.transform(X_test)

print(X_train_pre.shape, X_valid_pre.shape, X_test_pre.shape)


feature_names = preprocessor.get_feature_names_out()
pd.DataFrame(X_train_pre, columns=feature_names).head()


(189, 28) (40, 28) (41, 28)


,num__age,num__bp,num__cholesterol,num__max_hr,num__st_depression,cat__sex_0,cat__sex_1,cat__chest_pain_type_1,cat__chest_pain_type_2,cat__chest_pain_type_3,...,cat__slope_of_st_1,cat__slope_of_st_2,cat__slope_of_st_3,cat__number_of_vessels_fluro_0,cat__number_of_vessels_fluro_1,cat__number_of_vessels_fluro_2,cat__number_of_vessels_fluro_3,cat__thallium_3,cat__thallium_6,cat__thallium_7
0,-0.953118,0.398356,-0.234783,0.096647,-0.726389,1.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
1,-1.057125,-1.232039,-1.117283,1.186523,-0.906202,0.0,1.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
2,0.190954,-0.649755,-1.207796,0.532598,0.802024,0.0,1.0,1.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
3,-0.225072,-0.183928,0.195153,0.489002,-0.906202,0.0,1.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
4,1.023006,0.514813,1.507590,-0.731658,-0.726389,1.0,0.0,0.0,0.0,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0


In [86]:
from pathlib import Path
import joblib

models_dir = Path("../../models")
models_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(preprocessor, models_dir / "heart_preprocessor.joblib")


['../../models/heart_preprocessor.joblib']